# 04 — Evaluate (test scenarios) and export

Uses the **validation-selected** policy from notebook 03.
Runs reserved **sim_test** scenarios, sensitivity checks, export bundle, and save/load consistency.

Fill `MODEL_CARD_TEMPLATE.md` with real numbers only — never invent results.


In [ ]:
# === USER CONFIG ===
RUN_IN_COLAB = True
DRIVE_SUBDIR = "AdaptiveLearnSE/c4_runs"
RUN_DIR = None  # path to train_smoke_... folder; if None, pick latest under output root
C4_ROOT = None
EVAL_EPISODES = 20


In [ ]:
from pathlib import Path
import sys
import subprocess
import json

if C4_ROOT is None:
    C4_ROOT = Path("..").resolve()
sys.path.insert(0, str(C4_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(C4_ROOT / "requirements-training.txt")])

from training.paths import resolve_output_root, stage_dirs
from training.colab_bootstrap import mount_drive
from training.config import EnvConfig
from training.pipeline import load_simulator_from_artifacts, require_previous_stage
from training.evaluate import evaluate_on_test, sensitivity_to_assumptions, save_evaluation
from training.export_policy import export_bundle, inference_example_source, save_load_consistency_check
from training.provenance import read_json

if RUN_IN_COLAB:
    try:
        mount_drive()
    except Exception as e:
        print(e)

OUT = resolve_output_root(DRIVE_SUBDIR if RUN_IN_COLAB else None, prefer_drive=RUN_IN_COLAB)
DIRS = stage_dirs(OUT)
ARTIFACTS = require_previous_stage(DIRS["artifacts"] / "response_model.joblib", "01", "Run notebook 01.").parent

if RUN_DIR is None:
    cands = sorted(OUT.glob("train_*"), key=lambda p: p.stat().st_mtime)
    if not cands:
        raise FileNotFoundError("No train_* run dir found. Run notebook 03 first.")
    RUN_DIR = cands[-1]
RUN_DIR = Path(RUN_DIR)
selection = read_json(RUN_DIR / "selection_val.json")
print("Using RUN_DIR", RUN_DIR)
print("Selection", selection)


In [ ]:
algo = selection.get("selected")
if not algo:
    raise RuntimeError("No algorithm selected in selection_val.json")

model_path = None
comp = read_json(RUN_DIR / "comparison_val.json")
seed_key = sorted(comp["seeds"].keys())[0]
payload = comp["seeds"][seed_key].get(algo, {})
model_path = payload.get("path")

env_cfg = EnvConfig()
sim = load_simulator_from_artifacts(ARTIFACTS, env_cfg)

test_report = evaluate_on_test(
    sim,
    env_cfg,
    selected_algorithm=algo,
    model_path=model_path,
    episodes=EVAL_EPISODES,
    seed=123,
    scenario="sim_test",
)
sens = sensitivity_to_assumptions(
    sim,
    env_cfg,
    policy_name=algo,
    model_path=model_path,
    episodes=max(5, EVAL_EPISODES // 2),
    seed=99,
)
report = {"test": test_report, "sensitivity": sens, "selection": selection}
save_evaluation(report, DIRS["metrics"] / "final_test_report.json")
print(json.dumps(test_report["summary"], indent=2))
print(test_report["disclaimer"])


In [ ]:
prov = read_json(ARTIFACTS / "real_data_provenance.json")
train_cfg = read_json(RUN_DIR / "train_config.json")
export_dir = export_bundle(
    export_dir=DIRS["exports"] / "latest",
    simulator=sim,
    env_config=env_cfg,
    selected_algorithm=algo,
    model_path=model_path,
    preprocessing_dir=ARTIFACTS,
    dataset_provenance=prov,
    evaluation_summary=report,
    training_config=train_cfg,
)
print("Exported to", export_dir)
if model_path and algo in {"dqn", "ppo"}:
    print("Save/load check", save_load_consistency_check(Path(model_path), sim, env_cfg, algorithm=algo))
print("\nInference example:\n")
print(inference_example_source())
print("\nFill MODEL_CARD_TEMPLATE.md with the metrics above. Do not invent numbers.")
